# Original Classification Notebook — Cleaned

**Project:** NYC Taxi Trip Duration & Late-Risk Prediction  
**Purpose:** Model late-trip risk as an imbalanced classification problem and tune probability thresholds.

> This notebook is a cleaned portfolio version of the original modeling work. Outputs were cleared for GitHub readability. The project-level README contains the final summarized results and interpretation.


# Classification Models

In [ ]:
# 1. Imports
import pandas as pd

#### Load Data

In [ ]:
trip_df = pd.read_csv(r"taxi_data.csv",low_memory=False)
zone_df = pd.read_csv(r"taxi_zone_lookup.csv")

#### Convert to DateTime

In [ ]:
# Convert pickup & dropoff to datetime
trip_df["tpep_pickup_datetime"]  = pd.to_datetime(trip_df["tpep_pickup_datetime"],  errors="coerce")
trip_df["tpep_dropoff_datetime"] = pd.to_datetime(trip_df["tpep_dropoff_datetime"], errors="coerce")

#### Create Trip Duration in Minutes

In [ ]:
# Create trip duration in minutes
trip_df["trip_duration_min"] = (trip_df["tpep_dropoff_datetime"] - trip_df["tpep_pickup_datetime"]).dt.total_seconds() / 60

# See summary stats of trip_duration_min
trip_df["trip_duration_min"].describe()

In [ ]:
# Check suspicious durations
n_le0   = (trip_df["trip_duration_min"] <= 0).sum()
n_0_1   = ((trip_df["trip_duration_min"] > 0) & (trip_df["trip_duration_min"] <= 1)).sum()
n_gt180 = (trip_df["trip_duration_min"] > 180).sum()

n_le0, n_0_1, n_gt180

In [ ]:
# Drop trips with duration ≤ 0, 0–1 minute, or > 180 minutes
trip_df = trip_df[
    trip_df["trip_duration_min"].notna() &
    (trip_df["trip_duration_min"] > 1) &
    (trip_df["trip_duration_min"] <= 180)
].copy()

# Optional: check how many rows are left
trip_df.shape

In [ ]:
# See columns, types, and non-missing counts
trip_df.info()

In [ ]:
trip_df["passenger_count"].value_counts().sort_index()

In [ ]:
# Keep only trips with at least 1 passenger
trip_df = trip_df[trip_df["passenger_count"] >= 1].copy()

# Check new size
trip_df.shape

In [ ]:
# Find LocationIDs for JFK and LaGuardia (search by zone name)
jfk_rows = zone_df[zone_df["Zone"].str.contains("JFK", case=False, na=False)]
lga_rows = zone_df[zone_df["Zone"].str.contains("LaGuardia", case=False, na=False)]

jfk_rows, lga_rows

In [ ]:
jfk_ids = jfk_rows["LocationID"].tolist()
lga_ids = lga_rows["LocationID"].tolist()

jfk_ids, lga_ids

In [ ]:
# Prepare pickup zone info
pu_zone = zone_df[["LocationID", "Borough", "Zone"]].rename(
    columns={"LocationID": "PULocationID", "Borough": "PU_Borough", "Zone": "PU_Zone"}
)

# Add pickup borough/zone to trips
trip_df = trip_df.merge(pu_zone, on="PULocationID", how="left")

In [ ]:
# Prepare dropoff zone info
do_zone = zone_df[["LocationID", "Borough", "Zone"]].rename(
    columns={"LocationID": "DOLocationID", "Borough": "DO_Borough", "Zone": "DO_Zone"}
)

# Add dropoff borough/zone to trips
trip_df = trip_df.merge(do_zone, on="DOLocationID", how="left")

In [ ]:
# Boolean filters
is_manhattan_pickup = trip_df["PU_Borough"] == "Manhattan"
is_jfk_dropoff      = trip_df["DOLocationID"].isin(jfk_ids)
is_lga_dropoff      = trip_df["DOLocationID"].isin(lga_ids)

# Keep only Manhattan → (JFK or LGA) trips
airport_trips = trip_df[is_manhattan_pickup & (is_jfk_dropoff | is_lga_dropoff)].copy()

airport_trips.shape

In [ ]:
airport_trips

In [ ]:
# Label each trip as JFK or LGA based on DOLocationID
airport_trips["airport"] = "LGA"
airport_trips.loc[airport_trips["DOLocationID"].isin(jfk_ids), "airport"] = "JFK"

airport_trips["airport"].value_counts()

In [ ]:
# Hour of day (0–23)
airport_trips["pickup_hour"] = airport_trips["tpep_pickup_datetime"].dt.hour

# Day of week (0=Mon, 6=Sun)
airport_trips["pickup_dow"] = airport_trips["tpep_pickup_datetime"].dt.dayofweek

In [ ]:
airport_trips = airport_trips[airport_trips["trip_distance"] > 0].copy()
airport_trips.shape

In [ ]:
airport_trips.to_csv("taxi_clean_for_modeling.csv", index=False)
print("Saved taxi_clean_for_modeling.csv")

# Classification

## Baseline model - Median duration (% Late vs % Not Late) and Logistic Regression

In [ ]:
# Group by airport, pickup hour, and pickup day
group_cols = ["airport", "pickup_hour", "pickup_dow"]

# Calculate the median duration for each group
median_dur = (airport_trips.groupby(group_cols)["trip_duration_min"].median().reset_index().rename(columns={"trip_duration_min": "median_duration"}))
median_dur.head(3)

In [ ]:
# Add typical duration to each trip
airport_trips = airport_trips.merge(median_dur, on=group_cols, how="left")

# Threshold = 1.2 × median (20% longer than typical)
airport_trips["late_threshold"] = 1.2 * airport_trips["median_duration"]

# Trip is late if duration > threshold
airport_trips["is_late"] = (
    airport_trips["trip_duration_min"] > airport_trips["late_threshold"]
).astype(int)

# See class balance
airport_trips["is_late"].value_counts(), airport_trips["is_late"].value_counts(normalize=True)

In [ ]:
# 2. Predictions (classes and probabilities)
y_pred_lr  = log_clf.predict(X_test)              # 0/1 prediction
y_proba_lr = log_clf.predict_proba(X_test)[:, 1]  # P(late = 1)

In [ ]:
# 3. Evaluate logistic regression baseline
acc_lr  = accuracy_score(y_test, y_pred_lr)
prec_lr = precision_score(y_test, y_pred_lr)
rec_lr  = recall_score(y_test, y_pred_lr)
f1_lr   = f1_score(y_test, y_pred_lr)
auc_lr  = roc_auc_score(y_test, y_proba_lr)

acc_lr, prec_lr, rec_lr, f1_lr, auc_lr

## Random Forest Classifier based on trip distance, passenger count, pickup hour, pickup day, airport

### First trial 25% recall

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
import pandas as pd

feature_cols = ["trip_distance", "passenger_count", "pickup_hour", "pickup_dow", "airport"]

# Drop rows with missing features or labels
clf_df = airport_trips.dropna(subset=feature_cols + ["is_late"]).copy()

X = clf_df[feature_cols]
y = clf_df["is_late"]

# Turn airport (JFK/LGA) into dummy column airport_LGA
X = pd.get_dummies(X, columns=["airport"], drop_first=True)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y   # keep late/on-time ratio similar in train and test
)

In [ ]:
clf = RandomForestClassifier(
    n_estimators=500,
    random_state=42,
    n_jobs=-1
)

clf.fit(X_train, y_train)

In [ ]:
# Class (0/1) predictions
y_pred = clf.predict(X_test)

# Late probabilities
y_proba = clf.predict_proba(X_test)[:, 1]

acc_rf = accuracy_score(y_test, y_pred)
prec_rf = precision_score(y_test, y_pred)
rec_rf = recall_score(y_test, y_pred)
f1_rf = f1_score(y_test, y_pred)
auc_rf = roc_auc_score(y_test, y_proba)

acc_rf, prec_rf, rec_rf, f1_rf, auc_rf

### Setting class = "balanced" and trying different thresholds

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

# Random Forest that pays more attention to the minority class (late = 1)
clf_bal = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    n_jobs=-1,
    class_weight="balanced"   # <-- key change
)

clf_bal.fit(X_train, y_train)

In [ ]:
# Class predictions (0/1)
y_pred_bal = clf_bal.predict(X_test)

# Probabilities of being late
y_proba_bal = clf_bal.predict_proba(X_test)[:, 1]

# Metrics for late class
acc_bal  = accuracy_score(y_test, y_pred_bal)
prec_bal = precision_score(y_test, y_pred_bal)
rec_bal  = recall_score(y_test, y_pred_bal)
f1_bal   = f1_score(y_test, y_pred_bal)
auc_bal  = roc_auc_score(y_test, y_proba_bal)

acc_bal, prec_bal, rec_bal, f1_bal, auc_bal

In [ ]:
y_proba_bal = clf_bal.predict_proba(X_test)[:, 1]

In [ ]:
import numpy as np
from sklearn.metrics import precision_score, recall_score, f1_score

thresholds = [0.5, 0.4, 0.35, 0.3, 0.25, 0.2]

for t in thresholds:
    y_pred_t = (y_proba_bal >= t).astype(int)
    prec_t = precision_score(y_test, y_pred_t)
    rec_t  = recall_score(y_test, y_pred_t)
    f1_t   = f1_score(y_test, y_pred_t)
    print(f"threshold={t:.2f}  precision={prec_t:.3f}  recall={rec_t:.3f}  f1={f1_t:.3f}")

### CatBoost

In [ ]:
# Install if you don't have it
# (run this cell once; you can skip if already installed)
!pip install catboost

In [ ]:
from catboost import CatBoostClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score
)
import pandas as pd
import numpy as np

In [ ]:
# Make sure these time columns exist
airport_trips["pickup_hour"] = airport_trips["tpep_pickup_datetime"].dt.hour
airport_trips["pickup_dow"]  = airport_trips["tpep_pickup_datetime"].dt.dayofweek

feature_cols = ["trip_distance", "passenger_count", "pickup_hour", "pickup_dow", "airport", "payment_type"]

# Drop rows with missing features or label
clf_df = airport_trips.dropna(subset=feature_cols + ["is_late"]).copy()

X = clf_df[feature_cols]
y = clf_df["is_late"]

# Turn airport (JFK/LGA) into dummy numeric column
X = pd.get_dummies(X, columns=["airport"], drop_first=True)

X.head(), y.value_counts()

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y      # keep class balance similar in train/test
)

In [ ]:
# Weight for positive class (late = 1)
pos_weight = (y_train == 0).sum() / (y_train == 1).sum()
pos_weight

In [ ]:
cb = CatBoostClassifier(
    iterations=400,          # number of boosting steps
    depth=6,
    learning_rate=0.1,
    loss_function="Logloss",
    eval_metric="AUC",
    scale_pos_weight=pos_weight,   # <-- handle imbalance
    random_state=42,
    verbose=50               # print every 50 iterations
)

cb.fit(X_train, y_train, eval_set=(X_test, y_test), use_best_model=True)

In [ ]:
# Probabilities of being late
y_proba_cb = cb.predict_proba(X_test)[:, 1]

thresholds = [0.5, 0.4, 0.35, 0.3, 0.25, 0.2, 0.15, 0.1]

for t in thresholds:
    y_pred_t = (y_proba_cb >= t).astype(int)
    prec_t = precision_score(y_test, y_pred_t)
    rec_t  = recall_score(y_test, y_pred_t)
    f1_t   = f1_score(y_test, y_pred_t)
    print(f"threshold={t:.2f}  precision={prec_t:.3f}  recall={rec_t:.3f}  f1={f1_t:.3f}")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# From your printout
thresholds = np.array([0.5, 0.4, 0.35, 0.3, 0.25, 0.2, 0.15, 0.1])
precisions = np.array([0.299, 0.257, 0.242, 0.226, 0.207, 0.189, 0.183, 0.181])
recalls    = np.array([0.662, 0.826, 0.868, 0.915, 0.954, 0.983, 0.995, 0.999])

# Keep thresholds 0.50, 0.40, 0.30, 0.20  (indices 0,1,3,5)
keep = [0, 1, 3, 5]

thr_show = thresholds[keep]
prec_show = precisions[keep]
rec_show  = recalls[keep]

x = np.arange(len(thr_show))  # 0..3
width = 0.35

plt.figure(figsize=(6,4))

plt.bar(x - width/2, prec_show, width, label="Precision", color="#4E4E78")
plt.bar(x + width/2, rec_show,  width, label="Recall",    color="#F0DAA8")

plt.xticks(x, [f"{t:.2f}" for t in thr_show])
plt.ylim(0, 1)
plt.xlabel("Threshold for predicting 'late'")
plt.ylabel("Score")
plt.title("CatBoost – Precision vs Recall at Different Thresholds")
plt.legend()
plt.tight_layout()
plt.savefig("threshold_comparison.png", dpi=300)  # <- save
plt.show()

# Visualizations related to Classification

### Grouped bar chart – Logistic vs RF (Accuracy / Precision / Recall / F1)

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

# Use threshold 0.4 for CatBoost
cb_threshold = 0.4

y_pred_cb = (y_proba_cb >= cb_threshold).astype(int)

accuracy_cb  = accuracy_score(y_test, y_pred_cb)
precision_cb = precision_score(y_test, y_pred_cb)
recall_cb    = recall_score(y_test, y_pred_cb)
f1_cb        = f1_score(y_test, y_pred_cb)

# AUC uses probabilities, so it doesn't depend on the threshold
auc_cb = roc_auc_score(y_test, y_proba_cb)

accuracy_cb, precision_cb, recall_cb, f1_cb, auc_cb

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Logistic regression metrics
acc_lr  = accuracy_score(y_test, y_pred_lr)
prec_lr = precision_score(y_test, y_pred_lr)
rec_lr  = recall_score(y_test, y_pred_lr)
f1_lr   = f1_score(y_test, y_pred_lr)

# Random Forest metrics
acc_rf  = accuracy_score(y_test, y_pred)
prec_rf = precision_score(y_test, y_pred)
rec_rf  = recall_score(y_test, y_pred)
f1_rf   = f1_score(y_test, y_pred)

import numpy as np

all_metrics = {
    "acc_lr": acc_lr, "prec_lr": prec_lr, "rec_lr": rec_lr, "f1_lr": f1_lr,
    "acc_rf": acc_rf, "prec_rf": prec_rf, "rec_rf": rec_rf, "f1_rf": f1_rf,
    "accuracy_cb": accuracy_cb, "precision_cb": precision_cb,
    "recall_cb": recall_cb, "f1_cb": f1_cb,
}

for name, val in all_metrics.items():
    print(name, "->", type(val), np.shape(val))


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

metrics = ["Accuracy", "Precision", "Recall", "F1"]

lr_scores = [acc_lr,  prec_lr,  rec_lr,  f1_lr]
rf_scores = [acc_rf,  prec_rf,  rec_rf,  f1_rf]
cb_scores = [accuracy_cb, precision_cb, recall_cb, f1_cb]

x = np.arange(len(metrics))
width = 0.25

plt.figure(figsize=(7, 4))

plt.bar(x - width, lr_scores, width, label="Logistic Regression", color="#8F6185")
plt.bar(x,         rf_scores, width, label="Random Forest",       color="#F5B727")
plt.bar(x + width, cb_scores, width, label="CatBoost (t=0.4)",    color="#4C72B0")

plt.xticks(x, metrics)
plt.ylim(0, 1)
plt.ylabel("Score")
plt.title("Classification Model Comparison (Late-trip prediction)")
plt.legend()
plt.tight_layout()
plt.savefig("classification_comparison.png", dpi=300)  # <- save
plt.show()

### ROC curves – Logistic vs RF

In [ ]:
from sklearn.metrics import roc_curve, roc_auc_score
import matplotlib.pyplot as plt

# ROC points
fpr_lr, tpr_lr, _ = roc_curve(y_test, y_proba_lr)
fpr_rf, tpr_rf, _ = roc_curve(y_test, y_proba)   # use your RF prob var here
fpr_cb, tpr_cb, _ = roc_curve(y_test, y_proba_cb)

# AUCs
auc_lr = roc_auc_score(y_test, y_proba_lr)
auc_rf = roc_auc_score(y_test, y_proba)
auc_cb = roc_auc_score(y_test, y_proba_cb)

plt.figure(figsize=(6, 4))
plt.plot(fpr_lr, tpr_lr, label=f"Logistic (AUC={auc_lr:.3f})",   color="#8F6185")
plt.plot(fpr_rf, tpr_rf, label=f"Random Forest (AUC={auc_rf:.3f})", color="#F5B727")
plt.plot(fpr_cb, tpr_cb, label=f"CatBoost (AUC={auc_cb:.3f})",  color="#4C72B0")

plt.plot([0, 1], [0, 1], linestyle="--", color="gray")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curves – Late-trip classification")
plt.legend()
plt.tight_layout()
plt.savefig("roc_comparison.png", dpi=300)  # <- save
plt.show()

### Precision–Recall curves – Logistic vs RF

In [ ]:
from sklearn.metrics import precision_recall_curve, average_precision_score

# PR points
prec_lr, rec_lr_curve, _ = precision_recall_curve(y_test, y_proba_lr)
prec_rf, rec_rf_curve, _ = precision_recall_curve(y_test, y_proba)
prec_cb, rec_cb_curve, _ = precision_recall_curve(y_test, y_proba_cb)

# Average precision (area under PR curve)
ap_lr = average_precision_score(y_test, y_proba_lr)
ap_rf = average_precision_score(y_test, y_proba)
ap_cb = average_precision_score(y_test, y_proba_cb)

plt.figure(figsize=(6, 4))
plt.plot(rec_lr_curve, prec_lr, label=f"Logistic (AP={ap_lr:.3f})",   color="#8F6185")
plt.plot(rec_rf_curve, prec_rf, label=f"Random Forest (AP={ap_rf:.3f})", color="#F5B727")
plt.plot(rec_cb_curve, prec_cb, label=f"CatBoost (AP={ap_cb:.3f})",  color="#4C72B0")

plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Precision–Recall Curves – Late class")
plt.legend()
plt.tight_layout()
plt.savefig("pr_comparison.png", dpi=300)  # <- save
plt.show()